In [2]:
# Zachary Katz
# zachary_katz@mines.edu
# 07 May 2025
# Gigi Albers

# Imports
%load_ext autoreload
%autoreload 2

import util.plotting_helpers as plothelp
import matplotlib.pyplot as plt
import earthaccess
import xarray as xr
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pyproj import Transformer


# Setup earthaccess.
# Follows protocol in https://book.cryointhecloud.com/how_tos/background/earthdata.html
auth = earthaccess.login(strategy="netrc")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
def swot_granules(NUM_PAIRS: int = 1) -> tuple[list, list]:
    """Fetches granules in PAIRS (Before #1, After #1, Before #2, After #2, ...)."""
    before_list, after_list = [], []
    
    for i in range(NUM_PAIRS):
        print(f"\n--- Pair {i+1}/{NUM_PAIRS} ---")
        
        # BEFORE granule
        print(f"🟦 BEFORE Event {i+1}:")
        while True:
            granule_name = input("Granule name: ")
            start_date = input("Start date (YYYY-MM-DD): ")
            end_date = input("End date (YYYY-MM-DD): ")
            
            results = earthaccess.search_data(
                short_name="SWOT_L2_HR_Raster_D",
                granule_name=granule_name,
                temporal=(start_date, end_date),
            )
            
            if len(results) == 0:
                print("❌ No granules found! Try again.")
            else:
                print(f"✅ Found {len(results)} granules. Using the first one...")
                ds_before = xr.open_dataset(earthaccess.open(results)[0], engine="h5netcdf")
                before_list.append(ds_before)
                break
        
        # AFTER granule (same location, different time)
        print(f"\n🟥 AFTER Event {i+1}:")
        while True:
            granule_name = input("Granule name (same location, different time): ")
            start_date = input("Start date (YYYY-MM-DD): ")
            end_date = input("End date (YYYY-MM-DD): ")
            
            results = earthaccess.search_data(
                short_name="SWOT_L2_HR_Raster_D",
                granule_name=granule_name,
                temporal=(start_date, end_date),
            )
            
            if len(results) == 0:
                print("❌ No granules found! Try again.")
            else:
                print(f"✅ Found {len(results)} granules. Using the first one...")
                ds_after = xr.open_dataset(earthaccess.open(results)[0], engine="h5netcdf")
                after_list.append(ds_after)
                break
    
    return before_list, after_list

In [4]:
def average_wse(ds_list: list[xr.Dataset]) -> xr.DataArray:
    """Averages water surface elevation across multiple granules."""
    wse_list = [ds["wse"] for ds in ds_list]
    return sum(wse_list) / len(wse_list)



In [5]:
def get_wse_variable(ds):
    """Find the water surface elevation variable in the dataset."""
    # List of possible variable names for water surface elevation
    possible_names = [
        'wse',  # Try this first since you mentioned it
        'water_surface_height_above_geoid',  # Official SWOT variable name
        'height',
        'water_surface_height',
        'water_level'
    ]
    
    # Check for each possible name
    for name in possible_names:
        if name in ds:
            return ds[name]
    
    # If not found, show available variables
    print("Error: Could not find water surface elevation variable.")
    print("Available variables in dataset:", list(ds.variables.keys()))
    raise ValueError("Water surface elevation variable not found")


--- Pair 1/1 ---
🟦 BEFORE Event 1:


Granule name:  SWOT_L2_HR_Raster_250m_UTM49R_N_x_x_x_032_271_100F_20250507T165500_20250507T165521_PID0_04.nc
Start date (YYYY-MM-DD):  2025-05-07
End date (YYYY-MM-DD):  2025-05-07


✅ Found 1 granules. Using the first one...


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 271.48it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:00<00:00,  1.49it/s]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 10356.31it/s]



🟥 AFTER Event 1:


Granule name (same location, different time):  SWOT_L2_HR_Raster_250m_UTM49R_N_x_x_x_034_271_100F_20250618T102510_20250618T102531_PID0_01.nc
Start date (YYYY-MM-DD):  2025-06-18
End date (YYYY-MM-DD):  2025-06-18


✅ Found 1 granules. Using the first one...


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 1888.48it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 22.72it/s]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 13751.82it/s]


In [10]:
def create_comparison_plots(before_ds_list, after_ds_list):
    """Create 6-panel plot with dual colorbars"""
    fig = plt.figure(figsize=(50/2.54, 30/2.54))
    gs = fig.add_gridspec(2, 5, width_ratios=[0.05, 1, 1, 1, 0.05],
                         wspace=0.3, hspace=0.4,
                         left=0.08, right=0.92)

    first_wse_mesh = None
    first_diff_mesh = None

    for i in range(len(before_ds_list)):
        ds_before = before_ds_list[i]
        ds_after = after_ds_list[i]
        
        wse_before = ds_before["wse"] + ds_before["height_cor_xover"]
        wse_after = ds_after["wse"] + ds_after["height_cor_xover"]
        wse_diff = wse_after - wse_before
        
        ax_before = fig.add_subplot(gs[i, 1], projection=ccrs.PlateCarree())
        ax_after = fig.add_subplot(gs[i, 2], projection=ccrs.PlateCarree())
        ax_diff = fig.add_subplot(gs[i, 3], projection=ccrs.PlateCarree())
        
        # CHANGED: Now storing just the mesh
        mesh_before = plot_swot_data(ax_before, ds_before, f"Before {i+1}")
        mesh_after = plot_swot_data(ax_after, ds_after, f"After {i+1}")
        
        mesh_diff = ax_diff.pcolormesh(
            wse_diff.x.values,
            wse_diff.y.values,
            wse_diff.values,
            transform=ccrs.PlateCarree(),
            cmap="coolwarm",
            vmin=-2,
            vmax=2
        )
        ax_diff.set_title(f"Difference {i+1}")
        
        if i == 0:
            first_wse_mesh = mesh_before
            first_diff_mesh = mesh_diff
    
    plt.subplots_adjust(wspace=0.4, hspace=0.3)
    return fig

cbar = fig.colorbar(mesh1, cax=cax, orientation='vertical', location ='right', pad=0.5)
cbar.set_label(
    "Water Surface Elevation [m]",
    fontsize=12,
    labelpad=15,  # Space between colorbar and label
    ha='center',  # Horizontal alignment
    va='bottom',  # Vertical alignment
    rotation=90    # 0=horizontal, 90=vertical
)


fig.text(
    x=0.45,  # Left alignment (0=far left, 1=far right)
    y=0.90,   # Vertical position (1=top of figure)
    s=input("Main location: "),  # Your custom title text
    fontsize=20,
    color='black',
    va='top', # Vertical alignment
    ha='center' )  # Horizontal alignment

# Get your data
NUM_PAIRS = 1
before_ds_list, after_ds_list = swot_granules(NUM_PAIRS)

# Create and show plot
fig = create_comparison_plots(before_ds_list, after_ds_list)
plt.show()

NameError: name 'mesh1' is not defined

In [8]:
swot_ds

<xarray.Dataset> Size: 85MB
Dimensions:                  (x: 659, y: 658)
Coordinates:
  * x                        (x) float64 5kB 5.37e+05 5.372e+05 ... 7.015e+05
  * y                        (y) float64 5kB 5.87e+06 5.87e+06 ... 6.034e+06
Data variables: (12/39)
    crs                      object 8B ...
    longitude                (y, x) float64 3MB ...
    latitude                 (y, x) float64 3MB ...
    wse                      (y, x) float32 2MB ...
    wse_qual                 (y, x) float32 2MB ...
    wse_qual_bitwise         (y, x) float64 3MB ...
    ...                       ...
    load_tide_fes            (y, x) float32 2MB ...
    load_tide_got            (y, x) float32 2MB ...
    pole_tide                (y, x) float32 2MB ...
    model_dry_tropo_cor      (y, x) float32 2MB ...
    model_wet_tropo_cor      (y, x) float32 2MB ...
    iono_cor_gim_ka          (y, x) float32 2MB ...
Attributes: (12/49)
    Conventions:                   CF-1.7
    title:                         Level 2 KaRIn High Rate Raster Data Product
    source:                        Ka-band radar interferometer
    history:                       2025-06-06T02:59:43Z : Creation
    platform:                      SWOT
    references:                    V1.4.1
    ...                            ...
    mgrs_latitude_band:            U
    x_min:                         537000.0
    x_max:                         701500.0
    y_min:                         5869750.0
    y_max:                         6034000.0
    institution:                   CNES